# 04 — Statistical Analysis and Figures

This notebook contains the final analysis used for the 293-observation combined
SuperCam LIBS acoustic dataset.

It includes:

- regional summary statistics
- Kruskal–Wallis tests and epsilon-squared effect sizes
- Holm-corrected Dunn post hoc tests
- Crater Floor Maaz vs. Seitah comparisons
- Crater Floor member-level comparisons
- Spearman metric correlations
- PCA and PCA loadings
- final PCA, Kruskal–Wallis table, and regional boxplot figures

Internal dataset labels use `Delta`; figures display this as `Delta Front`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import kruskal
import scikit_posthocs as sp

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## Paths and analysis variables

In [ ]:
PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "LIBS_acoustic_meta_sheet_v4_293.csv"
)

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

metrics = [
    "Slope (dB/s)",
    "C2 (dB)",
    "Spectral Centroid (Hz)",
    "Spectral Bandwidth (Hz)",
    "Peak Frequency (Hz)",
    "Rolloff 85% (Hz)",
    "High Frequency Fraction",
    "High/Low Ratio",
]

region_order = ["Crater Floor", "Delta", "Upper Fan"]

palette = {
    "Crater Floor": "#0072B2",
    "Delta": "#E69F00",
    "Upper Fan": "#4DAF4A",
}

print("Dataset shape:", df.shape)
print("\nRegion counts:")
print(df["Region"].value_counts())

## Regional summary statistics

In [ ]:
regional_summary = (
    df.groupby("Region")[metrics]
      .agg(["count", "mean", "median", "std"])
      .round(2)
)

regional_summary

## Regional Kruskal–Wallis tests

The stored `Slope (dB/s)` values are negative. The report presents this quantity
as positive decay rate (`-Slope`). Reversing the sign does not change the
Kruskal–Wallis H statistic, p-value, or epsilon-squared effect size because the
rank ordering is simply reversed.

In [ ]:
kw_rows = []

for metric in metrics:
    crater = df.loc[df["Region"] == "Crater Floor", metric].dropna()
    delta = df.loc[df["Region"] == "Delta", metric].dropna()
    upper = df.loc[df["Region"] == "Upper Fan", metric].dropna()

    H, p = kruskal(crater, delta, upper)

    N = len(crater) + len(delta) + len(upper)
    k = 3

    epsilon_squared = (H - k + 1) / (N - k)

    display_metric = (
        "Decay Rate (dB/s)"
        if metric == "Slope (dB/s)"
        else metric
    )

    kw_rows.append({
        "Metric": display_metric,
        "Source Column": metric,
        "H": H,
        "p-value": p,
        "Epsilon Squared": epsilon_squared,
        "n": N,
    })

kw = (
    pd.DataFrame(kw_rows)
    .sort_values("Epsilon Squared", ascending=False)
    .reset_index(drop=True)
)

kw.to_csv(
    RESULTS_DIR / "regional_kruskal_wallis.csv",
    index=False,
)

kw[["Metric", "H", "p-value", "Epsilon Squared", "n"]].round(4)

## Dunn post hoc tests

In [ ]:
pairwise_results = {}
pairwise_rows = []

for metric in metrics:
    table = sp.posthoc_dunn(
        df,
        val_col=metric,
        group_col="Region",
        p_adjust="holm",
    )

    pairwise_results[metric] = table

    display_metric = (
        "Decay Rate (dB/s)"
        if metric == "Slope (dB/s)"
        else metric
    )

    pairwise_rows.append({
        "Metric": display_metric,
        "Crater Floor vs Delta":
            table.loc["Crater Floor", "Delta"],
        "Crater Floor vs Upper Fan":
            table.loc["Crater Floor", "Upper Fan"],
        "Delta vs Upper Fan":
            table.loc["Delta", "Upper Fan"],
    })

pairwise_summary = (
    pd.DataFrame(pairwise_rows)
    .set_index("Metric")
)

pairwise_summary.to_csv(
    RESULTS_DIR / "regional_dunn_holm.csv"
)

pairwise_summary.round(5)

In [ ]:
significance_summary = pairwise_summary.copy()

for column in significance_summary.columns:
    significance_summary[column] = (
        significance_summary[column]
        .map(lambda p: "yes" if p < 0.05 else "no")
    )

significance_summary

## Crater Floor: Maaz vs. Seitah

In [ ]:
cf = df[df["Region"] == "Crater Floor"].copy()

print("Formation counts:")
print(cf["Formation"].value_counts())

formation_rows = []

for metric in metrics:
    maaz = cf.loc[cf["Formation"] == "Maaz", metric].dropna()
    seitah = cf.loc[cf["Formation"] == "Seitah", metric].dropna()

    H, p = kruskal(maaz, seitah)

    N = len(maaz) + len(seitah)
    k = 2
    epsilon_squared = (H - k + 1) / (N - k)

    display_metric = (
        "Decay Rate (dB/s)"
        if metric == "Slope (dB/s)"
        else metric
    )

    formation_rows.append({
        "Metric": display_metric,
        "H": H,
        "p-value": p,
        "Epsilon Squared": epsilon_squared,
        "n": N,
    })

kw_formation = (
    pd.DataFrame(formation_rows)
    .sort_values("p-value")
    .reset_index(drop=True)
)

kw_formation.to_csv(
    RESULTS_DIR / "crater_floor_maaz_seitah_kruskal_wallis.csv",
    index=False,
)

kw_formation.round(4)

## Crater Floor: member-level comparisons

In [ ]:
print("Sample counts by member:")
print(cf["Member"].value_counts().sort_index())

members = sorted(cf["Member"].dropna().unique())

member_rows = []

for metric in metrics:
    groups = [
        cf.loc[cf["Member"] == member, metric].dropna()
        for member in members
    ]

    H, p = kruskal(*groups)

    N = sum(len(group) for group in groups)
    k = len(members)
    epsilon_squared = (H - k + 1) / (N - k)

    display_metric = (
        "Decay Rate (dB/s)"
        if metric == "Slope (dB/s)"
        else metric
    )

    member_rows.append({
        "Metric": display_metric,
        "Source Column": metric,
        "H": H,
        "p-value": p,
        "Epsilon Squared": epsilon_squared,
        "n": N,
    })

kw_members = (
    pd.DataFrame(member_rows)
    .sort_values("p-value")
    .reset_index(drop=True)
)

kw_members.to_csv(
    RESULTS_DIR / "crater_floor_member_kruskal_wallis.csv",
    index=False,
)

kw_members[
    ["Metric", "H", "p-value", "Epsilon Squared", "n"]
].round(4)

In [ ]:
member_dunn_tables = {}

for _, result in kw_members.loc[
    kw_members["p-value"] < 0.05
].iterrows():

    display_metric = result["Metric"]
    source_metric = result["Source Column"]

    posthoc = sp.posthoc_dunn(
        cf,
        val_col=source_metric,
        group_col="Member",
        p_adjust="holm",
    )

    member_dunn_tables[display_metric] = posthoc

    safe_name = (
        display_metric.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
    )

    posthoc.to_csv(
        RESULTS_DIR
        / f"member_dunn_{safe_name}.csv"
    )

    print(f"\n===== {display_metric} =====")
    print(posthoc.round(4))

## Spearman correlation between acoustic metrics

This was a supporting analysis rather than the main regional test.

In [ ]:
corr = df[metrics].corr(method="spearman")
corr.to_csv(RESULTS_DIR / "metric_spearman_correlation.csv")
corr.round(3)

## Principal component analysis

In [ ]:
analysis_df = (
    df[["Region"] + metrics]
    .dropna()
    .copy()
)

X_scaled = StandardScaler().fit_transform(
    analysis_df[metrics]
)

pca = PCA(n_components=2)
scores = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": scores[:, 0],
    "PC2": scores[:, 1],
    "Region": analysis_df["Region"].values,
})

loadings = pd.DataFrame(
    pca.components_.T,
    index=metrics,
    columns=["PC1", "PC2"],
)

loadings.to_csv(
    RESULTS_DIR / "pca_loadings.csv"
)

print(
    "Explained variance ratio:",
    pca.explained_variance_ratio_,
)

loadings.round(3)

## Figure 1 — PCA by region

In [ ]:
pc1_variance = 100 * pca.explained_variance_ratio_[0]
pc2_variance = 100 * pca.explained_variance_ratio_[1]

sns.set_theme(context="talk", style="ticks")

fig, ax = plt.subplots(figsize=(9.5, 7.2))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Region",
    hue_order=region_order,
    palette=palette,
    s=72,
    alpha=0.70,
    edgecolor="white",
    linewidth=0.45,
    ax=ax,
)

ax.axhline(
    0,
    color="0.55",
    linestyle="--",
    linewidth=0.9,
    alpha=0.7,
)
ax.axvline(
    0,
    color="0.55",
    linestyle="--",
    linewidth=0.9,
    alpha=0.7,
)

x_range = pca_df["PC1"].max() - pca_df["PC1"].min()
y_range = pca_df["PC2"].max() - pca_df["PC2"].min()

ax.set_xlim(
    pca_df["PC1"].min() - 0.06 * x_range,
    pca_df["PC1"].max() + 0.06 * x_range,
)
ax.set_ylim(
    pca_df["PC2"].min() - 0.08 * y_range,
    pca_df["PC2"].max() + 0.08 * y_range,
)

ax.set_xlabel(
    f"PC1 ({pc1_variance:.1f}% variance explained)",
    fontsize=13,
)
ax.set_ylabel(
    f"PC2 ({pc2_variance:.1f}% variance explained)",
    fontsize=13,
)
ax.set_title(
    "Principal Component Analysis of SuperCam LIBS Acoustic Metrics",
    fontsize=16,
    weight="bold",
    pad=16,
)

legend = ax.legend(
    title="Geological Region",
    frameon=True,
    loc="lower right",
    fontsize=11,
    title_fontsize=11,
)
legend.get_frame().set_alpha(0.95)
legend.get_frame().set_linewidth(0.7)

ax.grid(True, linewidth=0.6, alpha=0.22)
sns.despine(ax=ax)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "PCA_SuperCam_LIBS_Acoustic_Metrics.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

## Table figure — regional Kruskal–Wallis results

In [ ]:
table_df = kw[
    ["Metric", "H", "p-value", "Epsilon Squared"]
].copy()

table_df["H"] = table_df["H"].map(
    lambda x: f"{x:.2f}"
)

def format_p(p):
    if p < 0.001:
        exponent = int(np.floor(np.log10(p)))
        coefficient = p / (10 ** exponent)
        return rf"${coefficient:.2f} \times 10^{{{exponent}}}$"
    return f"{p:.4f}"

table_df["p-value"] = table_df["p-value"].map(format_p)
table_df["Epsilon Squared"] = (
    table_df["Epsilon Squared"]
    .map(lambda x: f"{x:.3f}")
)

table_df.columns = [
    "Metric",
    "H",
    "p-value",
    r"$\epsilon^2$",
]

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.axis("off")

tbl = ax.table(
    cellText=table_df.values,
    colLabels=table_df.columns,
    cellLoc="center",
    colLoc="center",
    loc="center",
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.4)

for (row, col), cell in tbl.get_celld().items():
    cell.set_linewidth(0.6)
    if row == 0:
        cell.set_text_props(weight="bold")
        cell.set_height(cell.get_height() * 1.15)

for i in range(1, len(table_df) + 1):
    tbl[(i, 0)].get_text().set_ha("left")

plt.title(
    "Kruskal–Wallis comparison of acoustic metrics by region (n = 293)",
    fontsize=11,
    pad=12,
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "Kruskal_Wallis_Table.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

## Figure 2 — regional boxplots

In [ ]:
plot_df = df.copy()
plot_df["Decay Rate (dB/s)"] = -plot_df["Slope (dB/s)"]

metrics_to_plot = [
    ("Spectral Bandwidth (Hz)", "Spectral Bandwidth (Hz)"),
    ("High Frequency Fraction", "High Frequency Fraction"),
    ("C2 (dB)", "C2 (dB)"),
    ("Decay Rate (dB/s)", "Decay Rate (dB/s)"),
]

sns.set_theme(context="talk", style="ticks")

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for ax, (column, ylabel) in zip(
    axes,
    metrics_to_plot,
):
    data = plot_df[["Region", column]].dropna()

    sns.boxplot(
        data=data,
        x="Region",
        y=column,
        order=region_order,
        hue="Region",
        hue_order=region_order,
        palette=palette,
        width=0.58,
        showfliers=False,
        linewidth=1.2,
        legend=False,
        ax=ax,
    )

    sns.stripplot(
        data=data,
        x="Region",
        y=column,
        order=region_order,
        hue="Region",
        hue_order=region_order,
        palette=palette,
        jitter=0.22,
        size=3.5,
        alpha=0.42,
        linewidth=0,
        legend=False,
        ax=ax,
    )

    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_xticks(range(3))
    ax.set_xticklabels(
        ["Crater Floor", "Delta Front", "Upper Fan"],
        fontsize=11,
    )
    ax.tick_params(axis="y", labelsize=10)
    ax.grid(axis="y", linewidth=0.6, alpha=0.22)
    sns.despine(ax=ax)

fig.suptitle(
    "Regional Variation in SuperCam LIBS Acoustic Metrics",
    fontsize=17,
    weight="bold",
    y=1.01,
)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "Regional_Acoustic_Metrics.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

## Expected regional results

With the retained 293-row dataset, the historical regional Kruskal–Wallis results
should reproduce approximately:

| Metric | H | p-value | epsilon-squared |
|---|---:|---:|---:|
| Spectral Bandwidth | 154.26 | 3.19e-34 | 0.525 |
| High Frequency Fraction | 81.41 | 2.10e-18 | 0.274 |
| C2 | 64.35 | 1.06e-14 | 0.215 |
| Decay Rate | 60.22 | 8.40e-14 | 0.201 |
| Spectral Centroid | 19.27 | 6.54e-05 | 0.060 |
| Peak Frequency | 13.52 | 1.16e-03 | 0.040 |
| Rolloff 85% | 5.73 | 0.0571 | 0.013 |
| High/Low Ratio | 5.43 | 0.0662 | 0.012 |